# Registration Pipeline

本 Notebook 介绍了 H&E 与 mIF（DAPI）病理图像之间从粗到精的三阶段配准流程：

| 阶段 | 操作 | 源文件 | 精度级别 |
|------|------|--------|----------|
| Step 1 | 主体方向对齐（旋转） | `coarse_registration/rotate_slide.py` | 切片级 |
| Step 2 | 缩略图级配准（刚性+非刚性） | `register.py` | 缩略图级 |
| Step 3 | Patch 级二次精细配准 | `extract_patches.py` | Patch 级（512×512） |

In [ ]:
import os
import sys
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import tifffile

Image.MAX_IMAGE_PIXELS = None
Image.OPEN_IGNORE_TRUNCATED_IMAGES = True

# 设置数据路径（请根据实际情况修改）
DATA_ROOT = "/path/to/data"
HE_PATH = os.path.join(DATA_ROOT, "he_rotated/xxx_rotated.tiff")
DAPI_PATH = os.path.join(DATA_ROOT, "mIF-Channel/xxx/xxx_DAPI.tiff")

---
## Step 1: Ensure the main body direction is consistent

**本步操作**：观察 HE 和 DAPI 切片的主体方向是否一致。如果方向不同（如一个横向、一个纵向），需要将 HE 切片旋转至与 DAPI 方向一致，否则后续配准无法成功。

**源文件**：`coarse_registration/rotate_slide.py`

### 1.1 可视化 HE 和 DAPI 的方向

缩放长边到 1024 像素生成缩略图，并排对比两张切片的主体朝向。

In [ ]:
def resize_longest_edge(img, max_edge=1024):
    w, h = img.size
    scale = max_edge / max(w, h)
    return img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)

he_img = Image.open(HE_PATH).convert("RGB")
dapi_img = Image.open(DAPI_PATH)
if dapi_img.mode == 'RGB' or dapi_img.mode == 'RGBA':
    dapi_img = dapi_img.convert('L')

he_thumbnail = resize_longest_edge(he_img)
dapi_thumbnail = resize_longest_edge(dapi_img)

plt.figure(figsize=(16, 8))
plt.subplot(1, 2, 1)
plt.imshow(np.array(he_thumbnail))
plt.axis('off')
plt.title("HE Thumbnail")

plt.subplot(1, 2, 2)
plt.imshow(np.array(dapi_thumbnail), cmap='gray')
plt.axis('off')
plt.title("DAPI Thumbnail")
plt.show()

### 1.2 旋转 HE 切片

**核心逻辑**（`rotate_slide.py` → `rotate_svs` 函数）：
1. 使用 PIL 的 `Image.rotate(angle, expand=True)` 旋转图像，`expand=True` 确保旋转后画布完整
2. 背景填充白色（`fillcolor=(255,255,255)`）避免黑色边缘干扰后续配准
3. 使用 `tifffile.imwrite` 保存为 BigTIFF 格式，支持超大文件

```shell
# 命令行用法
python coarse_registration/rotate_slide.py \
  --input_path /path/to/he_rotated/xxx.kfb \
  --output_dir /path/to/he_rotated/ \
  --level 1 \
  --angle 90 \
  --back_color white
```

In [ ]:
# ===== rotate_slide.py 中的核心函数 rotate_svs =====

def rotate_svs(input_path, output_path, level, angle, back_color='black'):
    import gc, tifffile as tiff

    # 读取图像（支持 kfb / svs / tif 格式）
    ext = os.path.splitext(input_path)[1]
    if ext == '.tif' or ext == '.tiff':
        level0 = Image.open(input_path).convert("RGB")
    # ... 其他格式的读取省略 ...

    # 旋转图像：expand=True 保证画布完整，fillcolor 控制背景色
    if back_color == 'black':
        rotated = level0.rotate(angle, expand=True)
    else:  # white
        rotated = level0.rotate(angle, expand=True, fillcolor=(255, 255, 255))

    # 保存为 BigTIFF（支持超大文件）
    rotated_np = np.array(rotated, dtype=np.uint8)
    tiff.imwrite(output_path, rotated_np,
                 bigtiff=True, photometric='rgb',
                 planarconfig='contig', tile=(512, 512),
                 compression=None)

    del rotated, rotated_np, level0
    gc.collect()

---
## Step 2: Thumbnail Level Registration

**本步操作**：在缩略图级别（长边 1024px）对 DAPI 和 HE 进行配准。先通过填充黑边使两张缩略图尺寸一致，然后用 SimpleITK 执行刚性配准（仿射变换）+ 非刚性微调（Demons），求出变换参数并保存为 `.pkl` 文件，供 Step 3 使用。

**源文件**：`register.py`

### 2.1 缩略图尺寸匹配（填充黑边）

**核心逻辑**（`register.py` → `pad_to_match_width` 函数）：

将 HE 和 DAPI 缩略图都 pad 到相同大小（取两者宽高的最大值），居中放置，剩余区域填充黑色。这样配准时不会因 resize 导致图像变形。

In [ ]:
# ===== register.py 中的核心函数 pad_to_match_width =====

def pad_to_match_width(img1, img2):
    """
    通过填充黑边使两个图像宽高一致，避免变形
    返回: (img1_padded, img2_padded, target_w, target_h)
    """
    h1, w1 = img1.shape[:2]
    h2, w2 = img2.shape[:2]
    target_w = max(w1, w2)
    target_h = max(h1, h2)

    # 创建黑色画布，居中放置原图
    img1_padded = np.zeros((target_h, target_w), dtype=img1.dtype)
    img2_padded = np.zeros((target_h, target_w), dtype=img2.dtype)

    y1_off = (target_h - h1) // 2
    x1_off = (target_w - w1) // 2
    y2_off = (target_h - h2) // 2
    x2_off = (target_w - w2) // 2

    img1_padded[y1_off:y1_off+h1, x1_off:x1_off+w1] = img1
    img2_padded[y2_off:y2_off+h2, x2_off:x2_off+w2] = img2

    return img1_padded, img2_padded, target_w, target_h

### 2.2 缩略图配准（刚性 + Demons 非刚性）

**核心逻辑**（`register.py` → `register_images` 函数）：
1. **转灰度 + 归一化**：RGB→灰度，clip 到 [0,1]
2. **高斯平滑**：sigma=1.5 降噪
3. **刚性配准**：SimpleITK AffineTransform，Mattes 互信息作为相似度度量，RegularStepGradientDescent 优化器
4. **非刚性微调**：DemonsRegistrationFilter 做局部变形校正
5. **保存变换参数**：将刚性变换、位移场、缩略图尺寸、原图尺寸等打包为 `.pkl`

In [ ]:
# ===== register.py 中的核心函数 register_images =====

import SimpleITK as sitk

def register_images(moving_rgb_np, fixed_rgb_np):
    """
    moving = DAPI, fixed = HE
    返回: {'Transformation', 'DisplacementField', 'RegisteredImage'}
    """
    # --- 1. 转灰度 + 归一化 ---
    moving_gray = np.dot(moving_rgb_np[...,:3], [0.2989, 0.5870, 0.1140]) if moving_rgb_np.ndim == 3 else moving_rgb_np
    fixed_gray  = np.dot(fixed_rgb_np[...,:3],  [0.2989, 0.5870, 0.1140]) if fixed_rgb_np.ndim == 3  else fixed_rgb_np
    moving_gray = np.clip(moving_gray / 255.0, 0, 1) if moving_gray.max() > 1.0 else moving_gray
    fixed_gray  = np.clip(fixed_gray  / 255.0, 0, 1) if fixed_gray.max()  > 1.0 else fixed_gray

    moving = sitk.GetImageFromArray((moving_gray * 255).astype(np.uint8))
    fixed  = sitk.GetImageFromArray((fixed_gray  * 255).astype(np.uint8))
    fixed.SetOrigin((0.0, 0.0));  fixed.SetSpacing((1.0, 1.0))
    moving.SetOrigin((0.0, 0.0)); moving.SetSpacing((1.0, 1.0))

    # --- 2. 高斯平滑降噪 ---
    fixed  = sitk.SmoothingRecursiveGaussian(fixed,  sigma=1.5)
    moving = sitk.SmoothingRecursiveGaussian(moving, sigma=1.5)

    # --- 3. 刚性配准（AffineTransform + Mattes Mutual Information）---
    initial_transform = sitk.AffineTransform(2)
    initial_transform.SetMatrix([1.05, 0.0, 0.0, 1.00])  # 允许略微缩放
    center_moving = np.array(moving.GetSize()) / 2.0
    center_fixed  = np.array(fixed.GetSize())  / 2.0
    initial_transform.SetTranslation((center_fixed - center_moving).tolist())

    registration_method = sitk.ImageRegistrationMethod()
    registration_method.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)
    registration_method.SetMetricSamplingStrategy(registration_method.RANDOM)
    registration_method.SetMetricSamplingPercentage(1.0)
    registration_method.SetInterpolator(sitk.sitkLinear)
    registration_method.SetOptimizerAsRegularStepGradientDescent(
        learningRate=1.5, minStep=1e-6, numberOfIterations=5000, gradientMagnitudeTolerance=1e-6)
    registration_method.SetOptimizerScalesFromPhysicalShift()
    registration_method.SetInitialTransform(initial_transform, inPlace=False)

    final_transform = registration_method.Execute(fixed, moving)

    # --- 4. 非刚性微调（Demons）---
    moving_registered = sitk.Resample(moving, fixed, final_transform, sitk.sitkLinear, 0.0)
    demons = sitk.DemonsRegistrationFilter()
    demons.SetNumberOfIterations(80)
    demons.SetStandardDeviations(1.0)
    displacement_field = demons.Execute(fixed, moving_registered)

    return {
        'Transformation': final_transform,
        'DisplacementField': sitk.DisplacementFieldTransform(displacement_field)
    }

### 2.3 执行配准并保存变换参数

将缩略图配准结果与原图尺寸信息一起打包为 `xxx_transform.pkl`，供 Step 3 patch 级配准使用。

In [ ]:
import pickle

# 加载缩略图
def load_thumbnail(path, max_long_side=1024):
    img = Image.open(path)
    w, h = img.size
    if max(w, h) > max_long_side:
        scale = max_long_side / max(w, h)
        img = img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
    return np.array(img)

he_thumb = load_thumbnail(HE_PATH)       # RGB
dapi_thumb = load_thumbnail(DAPI_PATH)   # 灰度 or RGB

# 填充到相同尺寸
dapi_padded, he_padded, thumb_w, thumb_h = pad_to_match_width(
    dapi_thumb if dapi_thumb.ndim == 2 else np.dot(dapi_thumb[...,:3], [0.2989, 0.5870, 0.1140]),
    he_padded if he_thumb.ndim == 2 else np.dot(he_thumb[...,:3], [0.2989, 0.5870, 0.1140])
)

# 执行配准
result = register_images(dapi_padded, he_padded)

# 获取原图尺寸
he_orig_size = Image.open(HE_PATH).size    # (width, height)
dapi_orig_size = Image.open(DAPI_PATH).size

# 保存变换参数
transform_data = {
    'Transformation': result['Transformation'],      # 刚性变换
    'DisplacementField': result['DisplacementField'],# 非刚性位移场
    'he_original_size': he_orig_size,                # HE 原图像素尺寸
    'dapi_original_size': dapi_orig_size,            # DAPI 原图像素尺寸
    'thumbnail_size': (thumb_w, thumb_h),            # 填充后缩略图尺寸
    'padding_used': True,                            # 标记使用了填充模式
    'he_thumb_original_size': (he_thumb.shape[1], he_thumb.shape[0]),
    'dapi_thumb_original_size': (dapi_thumb.shape[1], dapi_thumb.shape[0])
}

# pkl_path = os.path.join(OUTPUT_DIR, "xxx_transform.pkl")
# with open(pkl_path, 'wb') as f:
#     pickle.dump(transform_data, f)
print("✓ 配准完成，变换参数已准备就绪")

In [ ]:
# 可视化验证：叠加 HE 和配准后的 DAPI
# registered_dapi = result.get('RegisteredImage', dapi_padded)
plt.figure(figsize=(6, 6))
plt.imshow(he_padded, cmap='gray')
# plt.imshow(registered_dapi, alpha=0.3, cmap='Blues')
plt.title("Overlay: HE + Registered DAPI")
plt.axis('off')
plt.show()

---
## Step 3: Patch Level Registration

**本步操作**：遍历 DAPI 原图上的网格坐标，利用 Step 2 求得的变换参数将每个 mIF patch 坐标映射到 HE 原图坐标。由于缩略图级配准存在误差，对每个 patch 进行基于 Dice 相似度的**二次精细配准**：在 HE 上扩大搜索范围（±100px），找到与 DAPI 掩码最佳匹配的位置。

**源文件**：`extract_patches.py`

```shell
# 命令行用法
python extract_patches.py \
  --he_path /path/to/he_rotated/xxx_rotated.tiff \
  --transform_dir /path/to/mif_registered/xxx_channel_info/ \
  --output_dir /path/to/patches/ \
  --patch_size 512 --stride 512
```

### 3.1 坐标映射：mIF patch → HE 原图坐标

**核心逻辑**（`extract_patches.py` → `map_patch_mif_to_he` 函数）：

Step 2 生成的 pkl 包含 `dapi_thumb_original_size` 和 `he_thumb_original_size`（填充前的原始缩略图尺寸），用于正确计算填充偏移：

1. **mIF 原图坐标 → mIF 原始缩略图坐标**：用原始缩略图尺寸（填充前）计算 downsample 比例
2. **加上居中填充偏移** → 得到填充后缩略图坐标（与配准时一致）
3. **应用配准变换的逆变换**：将 mIF 缩略图坐标映射到 HE 缩略图坐标
4. **减去 HE 侧填充偏移** → 恢复到 HE 原始缩略图坐标
5. **HE 原始缩略图坐标 → HE 原图坐标**：根据 downsample 比例换算回来
6. **边界检查 + 二次配准**（可选）：扩大 100px 区域，搜索最佳匹配位置

In [ ]:
# ===== extract_patches.py 中的核心坐标映射逻辑（简化版）=====

def map_patch_mif_to_he(mif_x, mif_y, patch_size, transform,
                        thumb_size, dapi_orig_size, he_orig_size,
                        dapi_thumb_original_size, he_thumb_original_size):
    """
    将 mIF 原图上的 patch 坐标映射到 HE 原图坐标

    Step 2 的 pkl 中保存了原始缩略图尺寸（填充前，保持原图宽高比）和
    填充后尺寸 thumb_size。这里用原始尺寸计算 downsample 比例，
    加填充偏移后再应用变换，最后减去 HE 侧填充偏移还原到 HE 原图坐标。

    返回: HE 原图上的包围盒 (min_x, min_y, max_x, max_y)
    """
    # 1. 用原始缩略图尺寸（填充前）计算缩放比例
    mif_ds_x = dapi_orig_size[0] / dapi_thumb_original_size[0]
    mif_ds_y = dapi_orig_size[1] / dapi_thumb_original_size[1]
    he_ds_x  = he_orig_size[0] / he_thumb_original_size[0]
    he_ds_y  = he_orig_size[1] / he_thumb_original_size[1]

    # 2. mIF 原图坐标 → mIF 原始缩略图坐标
    x_thumb_orig = mif_x / mif_ds_x
    y_thumb_orig = mif_y / mif_ds_y
    w_thumb_orig = patch_size / mif_ds_x
    h_thumb_orig = patch_size / mif_ds_y

    # 3. 加上居中填充偏移 → 填充后缩略图坐标（与配准时一致）
    mif_pad_x = (thumb_size[0] - dapi_thumb_original_size[0]) // 2
    mif_pad_y = (thumb_size[1] - dapi_thumb_original_size[1]) // 2
    x_thumb = x_thumb_orig + mif_pad_x
    y_thumb = y_thumb_orig + mif_pad_y

    # 4. 四个角点（在填充后缩略图空间）
    corners = np.array([
        [x_thumb, y_thumb],
        [x_thumb + w_thumb_orig, y_thumb],
        [x_thumb, y_thumb + h_thumb_orig],
        [x_thumb + w_thumb_orig, y_thumb + h_thumb_orig]
    ])

    # 5. 应用逆变换（moving→fixed 方向）
    inverted = transform.GetInverse()
    transformed = np.array([inverted.TransformPoint((float(p[0]), float(p[1]))) for p in corners])

    # 6. 减去 HE 侧的居中填充偏移 → HE 原始缩略图坐标
    he_pad_x = (thumb_size[0] - he_thumb_original_size[0]) // 2
    he_pad_y = (thumb_size[1] - he_thumb_original_size[1]) // 2
    transformed[:, 0] -= he_pad_x
    transformed[:, 1] -= he_pad_y

    # 7. HE 缩略图坐标 → HE 原图坐标
    he_corners = np.empty_like(transformed)
    he_corners[:, 0] = transformed[:, 0] * he_ds_x
    he_corners[:, 1] = transformed[:, 1] * he_ds_y

    # 8. 计算包围盒
    min_x = max(0, int(np.floor(he_corners[:, 0].min())))
    min_y = max(0, int(np.floor(he_corners[:, 1].min())))
    max_x = min(he_orig_size[0], int(np.ceil(he_corners[:, 0].max())))
    max_y = min(he_orig_size[1], int(np.ceil(he_corners[:, 1].max())))

    return min_x, min_y, max_x, max_y

### 3.2 二次精细配准：基于 Dice 相似度的局部搜索

**核心逻辑**（`extract_patches.py` → `refine_patch_alignment` 函数）：
1. **提取苏木精通道**：对 HE 区域用 `rgb2hed` 分解，取 H 通道并 Otsu 二值化
2. **DAPI 二值化**：对 DAPI patch 做灰度转换 + Otsu 二值化
3. **GPU 批量搜索**：在扩大区域（±100px, 步长5）的所有候选位置，批量裁剪 → resize 到 512×512 → 计算 Dice 分数
4. **选最优位置**：取 Dice 最高的偏移作为最终配准结果

In [ ]:
# ===== extract_patches.py 中的核心二次配准函数（简化版）=====

import torch
import torch.nn.functional as F
from skimage.color import rgb2hed
from skimage.filters import threshold_otsu

def refine_patch_alignment(he_patch_large, dapi_patch, target_h, target_w,
                           search_range=100, prev_offset=(100, 100), dapi_size=512):
    """
    对 HE patch 做二次精细配准（GPU 加速）
    he_patch_large: 扩大后的 HE 区域（比 target 大）
    dapi_patch: 512×512 的 DAPI patch
    返回: (best_offset, he_final_512x512, best_dice)
    """
    # --- 1. HE 苏木精通道 + Otsu 二值化 ---
    hed = rgb2hed(he_patch_large.astype(float) / 255.0)
    h_channel = hed[:, :, 0]
    h_min, h_max = h_channel.min(), h_channel.max()
    if h_max > h_min:
        h_channel = (h_channel - h_min) / (h_max - h_min)
    h_threshold = threshold_otsu((h_channel * 255).astype(np.uint8))
    h_mask = (h_channel > h_threshold / 255.0).astype(np.uint8)

    # --- 2. DAPI 灰度 + Otsu 二值化 ---
    dapi_gray = (np.dot(dapi_patch[...,:3], [0.2989, 0.5870, 0.1140])
                 if dapi_patch.ndim == 3 else dapi_patch.copy())
    if dapi_gray.max() > 1.0:
        dapi_gray = dapi_gray / 255.0
    dapi_threshold = threshold_otsu((dapi_gray * 255).astype(np.uint8))
    dapi_mask = (dapi_gray > dapi_threshold / 255.0).astype(np.uint8)

    # --- 3. 移到 GPU，批量搜索 ---
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    h_gpu = torch.from_numpy(h_mask).float().to(device)
    dapi_gpu = torch.from_numpy(dapi_mask).float().to(device)

    # 以 prev_offset 为中心，search_range 为半径生成候选位置
    large_h, large_w = h_mask.shape
    positions = [(dy, dx)
                 for dy in range(max(0, prev_offset[1]-search_range),
                                 min(large_h-target_h, prev_offset[1]+search_range), 5)
                 for dx in range(max(0, prev_offset[0]-search_range),
                                 min(large_w-target_w, prev_offset[0]+search_range), 5)]

    # 批量裁剪 → resize → 计算 Dice
    dice_scores = torch.zeros(len(positions), device=device)
    sum_dapi = dapi_gpu.sum()
    for batch_start in range(0, len(positions), 128):
        batch = positions[batch_start:batch_start+128]
        crops = torch.zeros((len(batch), 1, target_h, target_w), device=device)
        for i, (dy, dx) in enumerate(batch):
            crops[i, 0] = h_gpu[dy:dy+target_h, dx:dx+target_w]
        crops_resized = F.interpolate(crops, size=(dapi_size, dapi_size),
                                      mode='bilinear', align_corners=False).squeeze(1)
        crops_resized = torch.round(crops_resized)
        intersections = (crops_resized * dapi_gpu).sum(dim=(1, 2))
        sum_h = crops_resized.sum(dim=(1, 2))
        dice_scores[batch_start:batch_start+len(batch)] = 2.0 * intersections / (sum_h + sum_dapi + 1e-8)

    # --- 4. 选择最优偏移 ---
    best_idx = torch.argmax(dice_scores).item()
    best_dy, best_dx = positions[best_idx]
    best_dice = dice_scores[best_idx].item()
    he_crop = he_patch_large[best_dy:best_dy+target_h, best_dx:best_dx+target_w]
    he_final = np.array(Image.fromarray(he_crop).resize((dapi_size, dapi_size), Image.LANCZOS))

    return (best_dx, best_dy), he_final, best_dice

### 3.3 完整的 Patch 提取流程

**核心逻辑**（`extract_patches.py` → `extract_patches_for_case` 函数）：

整个流程分为三个阶段：
- **阶段1**：网格扫描，过滤无效 HE（空白/黑暗/低纹理）和低 DAPI 掩码占比的 patch，保存有效坐标到 `patch_coords.json`
- **阶段2**：对有效坐标逐个执行坐标映射 + 二次配准，按 Dice 分数过滤（<0.1 跳过），保存 HE/mIF patch 图片
- **阶段3**：多线程写入磁盘

In [ ]:
# ===== extract_patches.py 的三阶段流程示意 =====

def extract_patches_for_case(he_path, transform_dir, output_dir,
                             patch_size=512, stride=512):
    """
    完整的三阶段 patch 提取流程（伪代码示意）
    """
    # 加载 Step 2 保存的变换参数（新格式 pkl，已标记 padding_used=True）
    transform_data = pickle.load(open(f"{transform_dir}/xxx_transform.pkl", 'rb'))
    rigid_transform = transform_data['Transformation']
    thumb_size = transform_data['thumbnail_size']          # 填充后缩略图尺寸
    dapi_orig_size = transform_data['dapi_original_size']
    he_orig_size = transform_data['he_original_size']
    # 从 pkl 读取原始缩略图尺寸（填充前），用于正确计算填充偏移
    dapi_thumb_original_size = transform_data['dapi_thumb_original_size']
    he_thumb_original_size = transform_data['he_thumb_original_size']

    he_img = Image.open(he_path)

    # === 阶段1: 扫描验证 ===
    valid_coords = []
    for mif_y in range(0, dapi_orig_size[1] - patch_size, stride):
        for mif_x in range(0, dapi_orig_size[0] - patch_size, stride):
            # 快速映射到 HE，验证是否有效（非空白、非黑暗、DAPI 掩码充足）
            he_patch, _, _, _ = map_patch_mif_to_he(
                mif_x, mif_y, patch_size, patch_size,
                he_img, rigid_transform,
                thumb_size, dapi_orig_size,
                thumb_size, he_orig_size,
                dapi_thumb_original_size, he_thumb_original_size,
                refine_alignment=False  # 阶段1不做二次配准
            )
            if he_patch is not None and is_valid(he_patch):
                valid_coords.append({'x': mif_x, 'y': mif_y})

    # === 阶段2: 带二次配准的批量提取 ===
    for coord in valid_coords:
        mif_x, mif_y = coord['x'], coord['y']
        dapi_patch = load_dapi_patch(mif_x, mif_y, patch_size)

        # 带二次配准的坐标映射
        he_patch, _, best_offset, dice_score = map_patch_mif_to_he(
            mif_x, mif_y, patch_size, patch_size,
            he_img, rigid_transform,
            thumb_size, dapi_orig_size,
            thumb_size, he_orig_size,
            dapi_thumb_original_size, he_thumb_original_size,
            refine_alignment=True, dapi_patch=dapi_patch
        )

        if dice_score >= 0.1:
            save_patch(he_patch, mif_patches, mif_x, mif_y)

    # === 阶段3: 多线程保存 ===
    # ThreadPoolExecutor 并行写入磁盘 ...